In [1]:
import nba_api.stats.endpoints
import requests
import json
import pandas as pd
import nba_api

In [2]:
# Get Timberwolves player game logs for the season
wolves_player_logs = nba_api.stats.endpoints.PlayerGameLogs(season_nullable='2025-26',team_id_nullable='1610612750').get_data_frames()[0]

wolves_player_logs

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT
0,2025-26,1630162,Anthony Edwards,Anthony,1610612750,MIN,Minnesota Timberwolves,0022500824,2026-02-22T00:00:00,MIN vs. PHI,...,86,42,471,42,50,4,22,1,34:08,1
1,2025-26,1630183,Jaden McDaniels,Jaden,1610612750,MIN,Minnesota Timberwolves,0022500824,2026-02-22T00:00:00,MIN vs. PHI,...,86,118,709,117,50,4,129,1,34:15,1
2,2025-26,203944,Julius Randle,Julius,1610612750,MIN,Minnesota Timberwolves,0022500824,2026-02-22T00:00:00,MIN vs. PHI,...,17,136,707,218,50,4,211,1,31:09,1
3,2025-26,1630245,Ayo Dosunmu,Ayo,1610612750,MIN,Minnesota Timberwolves,0022500824,2026-02-22T00:00:00,MIN vs. PHI,...,328,251,722,318,50,4,302,1,28:13,1
4,2025-26,1628978,Donte DiVincenzo,Donte,1610612750,MIN,Minnesota Timberwolves,0022500824,2026-02-22T00:00:00,MIN vs. PHI,...,328,274,623,378,50,4,302,1,27:35,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720,2025-26,204060,Joe Ingles,Joe,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,491,514,310,542,50,4,520,1,16:10,1
721,2025-26,1642389,Zyon Pullin,Zyon,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,491,557,449,628,50,4,607,1,10:19,1
722,2025-26,1631262,Jules Bernard,Jules,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,491,624,471,645,50,4,650,1,4:10,1
723,2025-26,1641803,Tristen Newton,Tristen,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,491,557,310,663,50,4,650,1,3:08,1


In [8]:
# Rank each player by FGA within each game (1 = most attempts)
wolves_player_logs['FGA_RANK_IN_GAME'] = wolves_player_logs.groupby('GAME_ID')['FGA'].rank(
    method='min', ascending=False
).astype(int)

# Games where Jaden McDaniels had the second-most FGA on the team
jaden_second_fga = wolves_player_logs[
    (wolves_player_logs['PLAYER_NAME'] == 'Jaden McDaniels') &
    (wolves_player_logs['FGA_RANK_IN_GAME'] == 2)
].copy()

wins_second_fga = (jaden_second_fga['WL'] == 'W').sum()
losses_second_fga = (jaden_second_fga['WL'] == 'L').sum()

print("Timberwolves record when Jaden McDaniels has the 2nd-most FGA on the team:")
print(f"  {wins_second_fga}-{losses_second_fga} ({jaden_second_fga.shape[0]} games)")
jaden_second_fga[['GAME_DATE', 'MATCHUP', 'WL', 'FGA', 'FGM', 'FG_PCT', 'PTS']].sort_values(by='GAME_DATE', ascending=False)

Timberwolves record when Jaden McDaniels has the 2nd-most FGA on the team:
  7-6 (13 games)


,GAME_DATE,MATCHUP,WL,FGA,FGM,FG_PCT,PTS
1,2026-02-22T00:00:00,MIN vs. PHI,L,12,5,0.417,19
23,2026-02-11T00:00:00,MIN vs. POR,W,13,7,0.538,21
78,2026-02-02T00:00:00,MIN @ MEM,L,14,11,0.786,29
155,2026-01-20T00:00:00,MIN @ UTA,L,16,7,0.438,18
166,2026-01-17T00:00:00,MIN @ SAS,L,14,7,0.500,23
177,2026-01-16T00:00:00,MIN @ HOU,L,14,5,0.357,15
230,2026-01-06T00:00:00,MIN vs. MIA,W,15,7,0.467,19
353,2025-12-14T00:00:00,MIN vs. SAC,W,19,7,0.368,21
379,2025-12-06T00:00:00,MIN vs. LAC,W,13,10,0.769,27
392,2025-12-04T00:00:00,MIN @ NOP,W,14,6,0.429,14


In [7]:
# Jaden's games with at least 12 FGA
jaden_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Jaden McDaniels'].copy()
jaden_12_plus_fga = jaden_logs[jaden_logs['FGA'] >= 14]

# Above 50% FG: use FGM/FGA to avoid float precision issues
above_50_pct = (jaden_12_plus_fga['FGM'] / jaden_12_plus_fga['FGA']) > 0.5
count_above_50 = above_50_pct.sum()

print("Jaden McDaniels: games with ≥12 FGA and shooting above 50% FG")
print(f"  {count_above_50} games (out of {len(jaden_12_plus_fga)} games with ≥12 FGA)")
jaden_12_plus_fga.assign(FG_PCT_ACTUAL=(jaden_12_plus_fga['FGM']/jaden_12_plus_fga['FGA']).round(3))[
    ['GAME_DATE', 'MATCHUP', 'WL', 'FGA', 'FGM', 'FG_PCT_ACTUAL', 'PTS']
]

Jaden McDaniels: games with ≥12 FGA and shooting above 50% FG
  7 games (out of 14 games with ≥12 FGA)


,GAME_DATE,MATCHUP,WL,FGA,FGM,FG_PCT_ACTUAL,PTS
78,2026-02-02T00:00:00,MIN @ MEM,L,14,11,0.786,29
90,2026-01-31T00:00:00,MIN @ MEM,W,14,8,0.571,20
155,2026-01-20T00:00:00,MIN @ UTA,L,16,7,0.438,18
166,2026-01-17T00:00:00,MIN @ SAS,L,14,7,0.500,23
177,2026-01-16T00:00:00,MIN @ HOU,L,14,5,0.357,15
220,2026-01-08T00:00:00,MIN vs. CLE,W,14,11,0.786,26
230,2026-01-06T00:00:00,MIN vs. MIA,W,15,7,0.467,19
353,2025-12-14T00:00:00,MIN vs. SAC,W,19,7,0.368,21
363,2025-12-12T00:00:00,MIN @ GSW,W,15,8,0.533,17
392,2025-12-04T00:00:00,MIN @ NOP,W,14,6,0.429,14
